[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/qem/qem.ipynb)

# Quadric Error Metrics

This example simplifies a mesh by collapsing edges into vertices placed where they disturb the surface least. Every triangle of the mesh lies in a plane, and moving a vertex off the planes of its triangles costs the sum of its squared distances to them. Collapsing an edge merges its two endpoints into one vertex that must serve the triangles of both, and the best place for it is where that summed cost is least: on a sharp ridge it stays on the ridge, pulled towards the sharper end.

In 3D projective geometric algebra, a plane is modelled as a vector and a point is modelled as an antivector (`Point`), so their regressive pairing `plane & point` gives the point's signed distance from a unit plane. A plane's dyad `plane * (plane & Point)` maps points to polar planes, and pairing that polar plane back with the point evaluates the squared distance. The error quadric of a vertex is the sum of its incident triangles' plane dyads, and collapsing an edge adds the quadrics of its two endpoints. Solving for a spatial direction offset `Direction` from the edge's midpoint finds the minimum at unit homogeneous weight and selects the minimizer nearest the midpoint when a flat patch or crease leaves directions free, following the quadric error metric of [Garland and Heckbert](https://www.cs.cmu.edu/~garland/quadrics/).

In homogeneous matrix notation each unit plane reads as a four-vector $\mathbf p = (n_x, n_y, n_z, d)^{\mathsf T}$, its plane dyad reads as the outer product $Q = \mathbf p\mathbf p^{\mathsf T}$, and the squared distance at $\mathbf x = (x, y, z, 1)^{\mathsf T}$ reads as the quadratic form $\mathbf x^{\mathsf T} Q\,\mathbf x$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA3D as ga
from examples import MODEL_DIR
from examples.util.animation import save_figure, show_animation
from examples.surfaces.qem import plumbing, render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
# A quadric: the polar plane of each point.
Quadric = ga.gatype((Plane, Point))
Spatial = ga.gatype.from_blades("x y z")            # Vector+
Direction = Spatial.dual().output_gatype            # Antivector-

## 1. A ridge and its triangles' planes

The patch has six vertices and six triangles: a ridge from a sharp corner a to a gentler end b, two faces flanking the ridge edge, two steep faces at the corner, and two shallow ones at the end. Each triangle's supporting plane is the normalized join of its three vertices.

In [ ]:
# The patch's vertices, a and b first, and its triangles, the two flanking the ridge edge first.
ridge_vertices = [
    [ 0.35,  0.0,   0.25],  # 0: vertex a (sharp corner apex)
    [-0.35,  0.0,   0.25],  # 1: vertex b (crease continuation)
    [ 0.0,   0.55, -0.15],  # 2: left base vertex
    [ 0.0,  -0.55, -0.15],  # 3: right base vertex
    [ 0.75,  0.0,  -0.15],  # 4: front tip (steep corner at a)
    [-0.85,  0.0,  -0.15],  # 5: back tip (gentle ramp at b)
]
faces = np.array([
    [0, 1, 2],  # 0: left flank (shared by edge a-b)
    [1, 0, 3],  # 1: right flank (shared by edge a-b)
    [0, 2, 4],  # 2: corner front-left (incident to a only)
    [0, 4, 3],  # 3: corner front-right (incident to a only)
    [1, 5, 2],  # 4: ramp back-left (incident to b only)
    [1, 3, 5],  # 5: ramp back-right (incident to b only)
])
# The faces incident to each endpoint of the ridge edge (a, b), the two flanking faces in both; the faces
# that survive the collapse.
incident_a = np.array([0, 1, 2, 3])
incident_b = np.array([0, 1, 4, 5])
surviving_faces = np.array([2, 3, 4, 5])

In [ ]:
vertices = mv.point(ridge_vertices)                                             # [vertices] Point-
face_planes = (vertices[faces[:, 0]] & vertices[faces[:, 1]] & vertices[faces[:, 2]]).normalized()   # [faces] Plane-

## 2. Collapsing the ridge edge

Each endpoint's quadric is the sum of its incident triangles' plane dyads, and the edge's quadric is their sum. The optimal replacement vertex has no spatial component in its polar plane. In the figure, the left and middle panels show each endpoint with its incident triangles and the ellipsoid where its quadric error stays small, while the right panel shows the collapsed mesh around the optimal vertex in green with the initial mesh faint behind it. The two flanking faces degenerate along the collapsed edge and vanish.

In [ ]:
def collapse_position(quadric: Quadric, midpoint: Point) -> Point:
    """The collapse position is the unit-weight minimum of the quadric, the one nearest the midpoint where flat directions leave a choice."""
    # A direction changes position without changing homogeneous weight. Only the spatial
    # part of the polar plane must vanish; its constant part is the residual minimum error.
    gradient = quadric(Direction).cast(Spatial)                                 # [...] Plane+ <- Direction-
    residual = quadric(midpoint).cast(Spatial)                                  # [...] Plane+
    correction = gradient.lstsq(-residual)                                      # [...] Direction-
    return midpoint + correction                                                # [...] Point-


# A unit plane measures a point's signed distance from it as `plane & point`; its dyad, a map from points to planes,
# gives the squared distance. Each endpoint's error quadric sums the dyads of its triangles' planes.
dyads = face_planes * (face_planes & Point)                                     # [faces] Plane+ <- Point-
quadric_a = dyads[incident_a].sum(axis=0)                                       # [] Plane+ <- Point-
quadric_b = dyads[incident_b].sum(axis=0)                                       # [] Plane+ <- Point-
# Contracting the edge adds the endpoints' quadrics; the best vertex is its unit-weight minimum.
edge_quadric = quadric_a + quadric_b                                            # [] Plane+ <- Point-
optimum = collapse_position(edge_quadric, (vertices[0] + vertices[1]) / 2)      # [] Point-
collapsed_vertices = concatenate([optimum.broadcast_to((2,)), vertices[2:]])    # [vertices] Point-
save_figure(render.draw_qem(vertices, faces, incident_a, incident_b, quadric_a, quadric_b, edge_quadric, optimum,
                            collapsed_vertices, surviving_faces), "qem_mesh_simplification");

In [ ]:
endpoints = vertices[:2]                                                        # [endpoints] Point-
print("the best vertex:\n", optimum.coefficient_table())
print("its error, against the endpoints':", (edge_quadric(optimum) & optimum).to_array(), (edge_quadric(endpoints) & endpoints).to_array())

## 3. Simplifying the bunny

The same construction reduces the Stanford bunny from 6,966 triangles to 250. Each merged vertex retains the sum of its predecessors' quadrics so that errors continue to refer to the initial surface. Each pass selects low-error contractions in disjoint neighbourhoods, keeping the mesh closed and preserving the orientation of every surviving face. Flat shading from a fixed camera shows the surviving facets around the ears and silhouette as the mesh coarsens.

In [ ]:
def simplify(vertices: Point, faces: np.ndarray, targets: np.ndarray) -> Iterator[tuple[Point, np.ndarray]]:
    """Yield a closed triangle mesh and its QEM contractions at descending face-count targets.

    The initial face planes remain the reference surface: every merged vertex retains the
    sum of its initial vertices' quadrics. Each pass contracts separate neighbourhoods,
    choosing the least errors first and preserving the topology and the sense of each face.
    """
    corners = vertices[faces]                                                   # [faces, 3] Point-
    planes = (corners[:, 0] & corners[:, 1] & corners[:, 2]).normalized()       # [faces] Plane-
    face_quadrics = planes * (planes & Point)                                   # [faces] Plane+ <- Point-
    quadrics = (face_quadrics[0] * 0).broadcast_to((len(vertices),))            # [vertices] Plane+ <- Point-
    quadrics = quadrics.at[faces.reshape(-1)].add(face_quadrics[:, None].broadcast_to((len(faces), 3)).reshape(-1))
    yield vertices, faces

    for target in targets:
        while len(faces) > target:
            topology = plumbing.Topology(faces, len(vertices))
            first, second = topology.edges.T
            edge_quadrics = quadrics[first] + quadrics[second]                  # [edges] Plane+ <- Point-
            midpoints = (vertices[first] + vertices[second]) / 2                # [edges] Point-
            optima = collapse_position(edge_quadrics, midpoints)                # [edges] Point-
            errors = edge_quadrics(optima) & optima                             # [edges] Scalar+

            # Test every surviving triangle in an edge's neighbourhood with that edge
            # contracted. Its plane must keep its sense and a nonzero area.
            old_corners = vertices[faces[topology.trial_faces]]                 # [trials, 3] Point-
            new_corners = old_corners.at[np.arange(len(old_corners)), topology.trial_corners].set(optima[topology.trial_edges])
            old_planes = old_corners[:, 0] & old_corners[:, 1] & old_corners[:, 2]  # [trials] Plane-
            new_planes = new_corners[:, 0] & new_corners[:, 1] & new_corners[:, 2]  # [trials] Plane-
            valid_trials = (old_planes.scalar_product(new_planes) > 0) & (new_planes.squared() > old_planes.squared() * 1e-12)
            chosen_edges = plumbing.select_edges(topology, errors, valid_trials, max(1, (len(faces) - target) // 2))

            # The chosen stars do not share changed faces, so their independent
            # contractions can be applied together without invalidating the face tests.
            vertices = vertices.at[first[chosen_edges]].set(optima[chosen_edges])
            quadrics = quadrics.at[first[chosen_edges]].set(edge_quadrics[chosen_edges])
            kept_vertices, faces = plumbing.contract(topology, chosen_edges)
            vertices, quadrics = vertices[kept_vertices], quadrics[kept_vertices]
        yield vertices, faces

In [ ]:
mesh_size = 2
face_targets = np.array([3500, 1800, 900, 450, 250])
duration_ms = 600
bunny_vertices, bunny_faces = plumbing.read_mesh(MODEL_DIR / "bunny.obj", mesh_size)
# The model's vertical axis is y; stand it upright along z.
upright = (mv.yz * (-np.pi / 4)).exp()                                          # [] Rotor+
bunny_vertices = upright >> bunny_vertices                                      # [vertices] Point-

states = simplify(bunny_vertices, bunny_faces, face_targets)
show_animation(render.animate_mesh(states, bunny_vertices), "qem_bunny", duration_ms,
               n_frames=len(face_targets) + 1, stepped=True)

In [ ]:
# checks
from examples.surfaces.qem import core

# Each face's plane passes through its three vertices.
np.testing.assert_allclose((face_planes[:, None] & vertices[faces]).to_array(), 0.0, atol=1e-12)

# The best vertex's polar plane is the plane at infinity; it stays on the ridge, between its middle and
# the sharp corner, and its error undercuts both endpoints'.
np.testing.assert_allclose((optimum - core.collapse_position(edge_quadric, (vertices[0] + vertices[1]) / 2)).kernel, 0.0, atol=1e-12)
assert edge_quadric(optimum).norm() < 1e-9
np.testing.assert_allclose((mv.y & optimum).to_array(), 0.0, atol=1e-12)
assert 0.0 < (mv.x & optimum) < (mv.x & vertices[0])
assert np.all((edge_quadric(optimum) & optimum) < (edge_quadric(endpoints) & endpoints))